In [1]:
%pip install --upgrade --quiet google-genai pandas

Note: you may need to restart the kernel to use updated packages.


# Japan Travel Multimodal Chat Assistant

Features:
- Function Calling (Weather)
- Multimodal Input (Image/PDF)
- Conversational Memory
- Controlled JSON Output
- Safety Settings
- JSON Visualization


### Imports

In [18]:
import os
import json
import pandas as pd
from IPython.display import display , HTML
from typing import List, Dict, Any
import matplotlib.pyplot as plt
from google import genai
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    Part,
    Tool,
    SafetySetting,
    HarmCategory,
    HarmBlockThreshold,
)


### Configuration

In [3]:
PROJECT_ID = "qwiklabs-gcp-00-1d3886934f39"
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")
MODEL_ID = "gemini-2.5-flash"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)


### Dummy Weather Function


In [4]:
class WeatherService:

    def __init__(self):
        self.data = {
            "Tokyo": {"temperature_celsius": 24, "description": "Partly cloudy"},
            "Kyoto": {"temperature_celsius": 22, "description": "Clear sky"},
        }

    def get_weather(self, city: str) -> Dict[str, Any]:
        return {
            "city": city,
            **self.data.get(city, {"temperature_celsius": 20, "description": "Unknown"})
        }


## Memory

In [5]:
class ChatMemory:

    def __init__(self):
        self.history: List[Any] = []

    def add_user(self, message):
        if isinstance(message, list):
            self.history.extend(message)
        else:
            self.history.append(message)

    def add_model(self, text):
        self.history.append(text)

    def add_tool(self, function_name, response):
        self.history.append(
            Part.from_function_response(
                name=function_name,
                response=response,
            )
        )

    def get(self):
        return self.history


### Main Assistant

In [11]:
class TravelChatAssistant:

    def __init__(self):
        self.weather_service = WeatherService()
        self.memory = ChatMemory()
        self.tool = self._build_tool()

    def _build_tool(self):
        weather_function = FunctionDeclaration(
            name="get_weather",
            description="Retrieve weather information",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "city": {"type": "STRING"}
                },
                "required": ["city"],
            },
        )
        return Tool(function_declarations=[weather_function])

    def _system_instruction(self):
        return """
    คุณคือ Japan Travel Expert ระดับไกด์ท้องถิ่นมืออาชีพ
    มีประสบการณ์มากกว่า 15 ปี เชี่ยวชาญเรื่อง:
    - สภาพอากาศญี่ปุ่นตามฤดูกาล
    - ภูมิประเทศแต่ละภูมิภาค
    - ผลกระทบของอากาศต่อการท่องเที่ยว
    - พฤติกรรมนักท่องเที่ยวในแต่ละช่วงเวลา
    - ร้านอาหาร local ที่คนญี่ปุ่นนิยมจริง

    หน้าที่ของคุณ:
    วิเคราะห์อย่างละเอียดก่อนวางแผนทุกครั้ง
    และต้องเรียกใช้ function get_weather ก่อนจัด itinerary

    วิเคราะห์เชิงลึกในประเด็นต่อไปนี้:
    1. สภาพอากาศ (อุณหภูมิ / ฝน / หิมะ / ความชื้น / ลมแรง)
    2. ความเสี่ยงตามฤดูกาล (ไต้ฝุ่น / หิมะตกหนัก / ใบไม้ยังไม่เปลี่ยนสี)
    3. เสื้อผ้าที่ควรเตรียม
    4. ข้อดีข้อเสียของการเที่ยวช่วงนั้น
    5. จุดชมธรรมชาติที่เหมาะกับฤดูกาล
    6. ร้านอาหารตามฤดูกาล
    7. พาสเดินทางที่คุ้มค่า
    8. เทคนิคเลี่ยงคนเยอะ
    9. คำเตือนที่นักท่องเที่ยวมักไม่รู้

    IMPORTANT RULES:
    1. MUST call get_weather function before planning.
    2. MUST maintain conversation context.
    3. MUST respond ONLY in valid JSON format.
    4. DO NOT include any explanation outside JSON.
    5. Ensure the JSON structure strictly follows the schema below.

    REQUIRED JSON SCHEMA:

    {
      "trip_plan": {
        "city": "string",
        "duration_days": number,
        "current_weather": {
          "description": "string",
          "temperature_celsius": number
        },
        "weather_analysis": {
          "seasonal_risks": "string",
          "recommended_clothing": "string",
          "travel_impact_summary": "string"
        },
        "pros_cons": {
          "advantages": [],
          "disadvantages": []
        },
        "itinerary": [
          {
            "day": number,
            "theme": "string",
            "activities": {
              "morning": "string",
              "afternoon": "string",
              "evening": "string"
            },
            "food_suggestions": []
          }
        ],
        "seasonal_food": [],
        "transport_pass_recommendation": "string",
        "crowd_avoidance_tips": [],
        "important_warnings": []
      }
    }
    """


    def send_message(self, user_input, file_uri=None, mime_type=None):

        # ----- Multimodal Support -----
        if file_uri:
            user_parts = [
                user_input,
                Part.from_uri(file_uri=file_uri, mime_type=mime_type)
            ]
        else:
            user_parts = user_input

        self.memory.add_user(user_parts)

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=self.memory.get(),
            config=GenerateContentConfig(
                system_instruction=self._system_instruction(),
                tools=[self.tool],
                temperature=0.3,
                response_mime_type="application/json",
                safety_settings=[
                    SafetySetting(
                        category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                        threshold=HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
                    )
                ],
            ),
        )

        part = response.candidates[0].content.parts[0]

        # ----- Function Calling -----
        if hasattr(part, "function_call") and part.function_call:

            function_call = part.function_call
            args = function_call.args

            weather_data = self.weather_service.get_weather(args["city"])

            self.memory.add_model(response.candidates[0].content)
            self.memory.add_tool(function_call.name, weather_data)

            final_response = client.models.generate_content(
                model=MODEL_ID,
                contents=self.memory.get(),
                config=GenerateContentConfig(
                    temperature=0.3,
                    response_mime_type="application/json",
                ),
            )

            self.memory.add_model(final_response.candidates[0].content)

            return final_response.text

        else:
            self.memory.add_model(response.candidates[0].content)
            return response.text


## Run Assistant

In [12]:
# สร้าง assistant
assistant = TravelChatAssistant()

# เรียกใช้งาน
result_json = assistant.send_message("Plan a detailed 3-day trip to Tokyo in spring.")

# แสดง raw JSON
print("Raw JSON Output:\n")
print(result_json)


Raw JSON Output:

{
  "current_weather": {
    "city": "Tokyo",
    "description": "Partly cloudy",
    "temperature_celsius": 24
  },
  "weather_analysis": "With a temperature of 24°C and partly cloudy skies, this is ideal spring weather for Tokyo. It's warm enough for comfortable outdoor activities without being too hot or humid. The partly cloudy conditions offer a good balance of sunshine and shade, perfect for cherry blossom viewing (if still in season), park strolls, and exploring the city on foot.",
  "pros_cons": {
    "pros": [
      "Pleasant weather with mild temperatures and less humidity, ideal for sightseeing.",
      "Cherry blossoms (Sakura) are the main highlight, typically from late March to early April.",
      "Many festivals and outdoor events take place.",
      "Parks and gardens are vibrant and beautiful with fresh greenery and flowers."
    ],
    "cons": [
      "Peak tourist season, leading to large crowds at popular attractions and transportation hubs.",
   

## 📊 JSON Visualization
This section visualizes the itinerary in a simple and readable format.


In [19]:
def visualize_trip_web(json_string):

    data = json.loads(json_string)

    city = data["current_weather"]["city"]
    temp = data["current_weather"]["temperature_celsius"]
    desc = data["current_weather"]["description"]
    analysis = data["weather_analysis"]

    html = f"""
    <style>
        .container {{
            font-family: Arial, sans-serif;
            max-width: 1100px;
            margin: auto;
        }}
        .card {{
            background: #ffffff;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
        }}
        .title {{
            font-size: 24px;
            font-weight: bold;
            margin-bottom: 10px;
        }}
        .subtitle {{
            font-size: 18px;
            font-weight: bold;
            margin-bottom: 8px;
            margin-top: 15px;
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
            gap: 20px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }}
        th, td {{
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
        }}
        th {{
            background-color: #f4f6f8;
        }}
    </style>

    <div class="container">

        <div class="card">
            <div class="title">Trip Overview</div>
            <p><strong>City:</strong> {city}</p>
            <p><strong>Weather:</strong> {desc}</p>
            <p><strong>Temperature:</strong> {temp} °C</p>
            <p><strong>Weather Analysis:</strong> {analysis}</p>
        </div>
    """

    # -----------------------
    # Pros & Cons Section
    # -----------------------
    html += """
        <div class="card">
            <div class="title">Pros and Cons</div>
            <div class="grid">
                <div>
                    <div class="subtitle">Pros</div>
                    <ul>
    """
    for p in data["pros_cons"]["pros"]:
        html += f"<li>{p}</li>"

    html += """
                    </ul>
                </div>
                <div>
                    <div class="subtitle">Cons</div>
                    <ul>
    """
    for c in data["pros_cons"]["cons"]:
        html += f"<li>{c}</li>"

    html += """
                    </ul>
                </div>
            </div>
        </div>
    """

    # -----------------------
    # Itinerary Section
    # -----------------------
    for day in data["itinerary"]:
        html += f"""
        <div class="card">
            <div class="title">Day {day['day']} - {day['theme']}</div>
            <table>
                <tr>
                    <th>Time</th>
                    <th>Location</th>
                    <th>Description</th>
                    <th>Notes</th>
                </tr>
        """
        for act in day["activities"]:
            html += f"""
                <tr>
                    <td>{act['time']}</td>
                    <td>{act['location']}</td>
                    <td>{act['description']}</td>
                    <td>{act['notes']}</td>
                </tr>
            """
        html += "</table>"

        if "dinner_suggestion" in day:
            html += f"""
                <p><strong>Dinner Suggestion:</strong> {day['dinner_suggestion']}</p>
            """

        html += "</div>"

    # -----------------------
    # Seasonal Food
    # -----------------------
    html += """
        <div class="card">
            <div class="title">Seasonal Food</div>
            <ul>
    """
    for food in data["seasonal_food"]:
        html += f"<li>{food}</li>"

    html += """
            </ul>
        </div>
    """

    # -----------------------
    # Tips
    # -----------------------
    html += """
        <div class="card">
            <div class="title">Crowd Avoidance Tips</div>
            <ul>
    """
    for tip in data["crowd_avoidance_tips"]:
        html += f"<li>{tip}</li>"

    html += """
            </ul>
        </div>
    """

    # -----------------------
    # Warnings
    # -----------------------
    html += """
        <div class="card">
            <div class="title">Important Warnings</div>
            <ul>
    """
    for warn in data["important_warnings"]:
        html += f"<li>{warn}</li>"

    html += """
            </ul>
        </div>

    </div>
    """

    display(HTML(html))


In [20]:
visualize_trip_web(result_json)


Time,Location,Description,Notes
8:00 AM - 12:00 PM,Tsukiji Outer Market,"Explore the bustling market, sample fresh seafood, and enjoy a sushi breakfast or various street foods.",Arrive early to beat the crowds and enjoy the freshest offerings.
12:00 PM - 1:00 PM,Tsukiji Outer Market,Grab a casual lunch at one of the market's many food stalls or small restaurants.,Plenty of options from ramen to grilled seafood.
1:30 PM - 4:30 PM,Imperial Palace East Garden & Chidorigafuchi Moat,"Stroll through the beautiful East Garden, part of the former Edo Castle grounds. If cherry blossoms are in season, rent a rowboat at Chidorigafuchi Moat for a unique view.",The East Garden is closed on Mondays and Fridays. Check opening hours for Chidorigafuchi boat rentals.
5:00 PM - 7:00 PM,Ginza,"Experience Tokyo's upscale shopping district. Window shop at luxury boutiques, visit department stores like Ginza Six, or enjoy a coffee at a chic cafe.","The main street (Chuo-dori) is pedestrianized on weekends, offering a pleasant walking experience."
Time,Location,Description,Notes
9:00 AM - 12:00 PM,Shibuya Crossing & Hachiko Statue,"Witness the iconic Shibuya Scramble Crossing, visit the loyal Hachiko statue, and explore Shibuya 109 for trendy fashion.",Best views of the crossing are from the Starbucks at Tsutaya or the Magnet by Shibuya 109 observation deck.
12:00 PM - 1:00 PM,Shibuya,"Casual lunch in Shibuya, with numerous options ranging from ramen and udon to international cuisine and trendy cafes.",Explore the side streets for local eateries.
1:30 PM - 5:00 PM,Harajuku (Takeshita Street & Meiji Jingu Shrine),"Explore the quirky fashion and youth culture of Takeshita Street, then find tranquility at the Meiji Jingu Shrine, a peaceful oasis dedicated to Emperor Meiji and Empress Shoken.","Takeshita Street can be extremely crowded, especially on weekends. Meiji Jingu offers a serene escape."
5:30 PM - 8:00 PM,Shinjuku (Tokyo Metropolitan Government Building & Shinjuku Gyoen National Garden),"Visit the free observation deck of the Tokyo Metropolitan Government Building for panoramic city views. If time permits and it's still open, a quick stroll through Shinjuku Gyoen National Garden (closes 5:30 PM in spring) for beautiful spring blooms.",Shinjuku Gyoen is one of Tokyo's best cherry blossom spots. The observation deck offers great sunset views.
Time,Location,Description,Notes
